### 5. Text Classification or Sentimental Analysis Proejct
Classify the message whether it is spam or ham

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('SMSSpamCollection', sep='\t', names=['class', 'message'])
df

,class,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [3]:
df["class"].unique()

array(['ham', 'spam'], dtype=object)

In [4]:
df["class"].value_counts()

class
ham     4825
spam     747
Name: count, dtype: int64

In [5]:
df.shape

(5572, 2)

**Data Understanding**

In [6]:
df.columns.tolist()

['class', 'message']

**Data Exploration**

In [7]:
df["class"].unique().tolist()

['ham', 'spam']

In [9]:
df["message"].nunique()

5169

In [10]:
## Checking no. of unique values in each column
df.isnull().sum()

class      0
message    0
dtype: int64

In [13]:
df[df["class"].isnull()]
# If any record has message but o/p label is missing, in this case fill the label
# by reading the message and assigning the label accordingly

,class,message


In [11]:
# to remove missing values
df = df.dropna()

#### Transform
##### Text Cleaning
One of the main reason is to reduce the input columns
- Remove Punctuation
- Remove Stopwords
- Stemming

In [14]:
import nltk
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

# Copora      Corpus
# sentence  : multiple words
# paragraph : multiple sentences
# document   : multiple paragraphs

corpus = [] # corpus is a collection of text documents. 
# Here we will create a list of all the messages after preprocessing them.
for i in range(0, len(df)):
    review = re.sub('[^a-zA-Z]', ' ', df['message'][i])
    review = review.lower()
    review = review.split()
    review = [ps.stem(word) for word in review if not word in set(stopwords.words('english'))]
    review = ' '.join(review)
    corpus.append(review)

In [15]:
corpus[:5]

['go jurong point crazi avail bugi n great world la e buffet cine got amor wat',
 'ok lar joke wif u oni',
 'free entri wkli comp win fa cup final tkt st may text fa receiv entri question std txt rate c appli',
 'u dun say earli hor u c alreadi say',
 'nah think goe usf live around though']

##### Vectorization
- **For input variable** : Count Vectorization
- **For output variable/label** : Label Encoding

**Count Vectorization**

In [16]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=None)
X = cv.fit_transform(corpus).toarray()
X
# Here we are using label encoder to convert the categorical labels into numerical labels
# This is done because machine learning algorithms work better with numerical data

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(5572, 6296))

In [17]:
X.shape

(5572, 6296)

**Label Encoding**

In [18]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

df["class"] = le.fit_transform(df["class"])
df["class"]

0       0
1       0
2       1
3       0
4       0
       ..
5567    1
5568    0
5569    0
5570    0
5571    0
Name: class, Length: 5572, dtype: int64

In [19]:
y = df["class"]

**Train Test Split**

In [20]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=0)

**Train the machine using ML Algorithm with train data**

Multicategory NaiveBayes

In [ ]:
from sklearn.naive_bayes import MultinomialNB
model = MultinomialNB()
model.fit(X_train, y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


**Evaluate your machine**
- train accuracy
- test accuracy
- cross validation score

In [22]:
ypred_train = model.predict(X_train)

from sklearn.metrics import accuracy_score
accuracy_score(y_train, ypred_train)

0.9921471842046219

In [23]:
ypred_test = model.predict(X_test)

from sklearn.metrics import accuracy_score
accuracy_score(y_test, ypred_test)

0.979372197309417

In [24]:
from sklearn.model_selection import cross_val_score
cross_val_score(model, X, y, cv=5).mean()

# Above code is for cross validation. It is used to evaluate the 
# performance of the model on unseen data. It is done by splitting 
# the data into k folds and training the model on k-1 folds and testing it 
# on the remaining fold. This process is repeated k times and the average 
# accuracy is taken as the final accuracy of the model.

np.float64(0.9768484272729469)

**Model Selection**
- Condition 1 : test accuracy = Train Accuracy
- Condition 2 : test accuracy = cross validation score

- Model said to be a good model, if its satisfy above 2 conditions
- if model fails to satisfy any Condition, Model said to be a bad model
- If Model have train accuracy > test accuracy, it is called as **Overfitting Problem**
    - Solution : reduce the no.of columns (dimension reduction = reduce the no.of input variables/words)
- If Model have train accuracy < test accuracy, it is called as **Underfitting Problem**
    - Solution : add more data

#### Prediction on New Data

In [25]:
input_mail = "You have wona lottery of 1cr"

**Load the Data**

**Preprocessing the Data**

In [26]:
# Text cleaning and preprocessing
corpus = []
rp = re.sub('[^a-zA-Z]', ' ', input_mail)
rp = rp.lower()
rp = rp.split()
rp = [ps.stem(word) for word in rp if not word in set(stopwords.words('english'))]
rp = ' '.join(rp)
corpus.append(rp)

# Text vectorization
X_input = cv.transform(corpus).toarray()

In [ ]:
X_input.shape
# 1 , 6296 # Here 1 is the number of messages and 6296 is the number of features (unique words in the corpus)
# future input will also have 6296 features but some of them will be 0 and some will be 1 depending on the 
# presence of the word in the message

(1, 6296)

**Prediction**

In [27]:
pred = model.predict(X_input)
if pred[0] == 1:
    print("Spam")
else:
    print("Not Spam")

Spam
